In [1]:
import numpy as np
import random as r

In [2]:
def griewank(x, numDim = 10):
    prodTerm = 1
    for i in range(len(x)):
        prodTerm *= np.cos(x[i] / np.sqrt(i + 1))
    sumTerm = 0
    for i in range(len(x)):
        sumTerm += (x[i] ** 2) / 4000
    return(sumTerm - prodTerm + 1)
ranges = [[-600, 600]] * 10

allowedIterations = 100

In [3]:
class RandomSearch:
    def __init__(
        self,
        ranges
    ):
        self.ranges = ranges
        
    def generateRandomHypothesis(self):
        output = []
        for x in self.ranges:
            output.append(r.uniform(x[0], x[1]))
        return(output)
    
    def optimize(self,
                objectiveFunction,
                numIterations = allowedIterations):
        bestX = None
        bestY = -float("inf")
        for i in range(numIterations):
            newX = self.generateRandomHypothesis()
            newY = objectiveFunction(newX)
            if newY > bestY:
                bestX = newX
                bestY = newY
        return(bestX)
rs = RandomSearch(ranges)
bestX = rs.optimize(griewank)
print(bestX)
print(griewank(bestX))

[-435.04125953358215, -442.52399177619856, -542.7805640542919, -574.8458009708669, -475.50437808508394, -466.72537842685057, 433.60446663586094, -400.6809216781502, 389.0117077770543, -439.4623575468938]
537.7747271285698


In [4]:
class SimulatedAnnealing:
    def __init__(
        self,
        ranges
    ):
        self.ranges = ranges
    
    def optimize(self, objectiveFunction, numIterations = allowedIterations, standardDeviation = 10, coolingFactor = 0.9):
        currentX = [np.mean(x) for x in self.ranges]
        currentY = objectiveFunction(currentX)
        heat = 1
        for i in range(numIterations):
            heat = heat * coolingFactor
            newX = [r.gauss(x, standardDeviation) for x in currentX]
            newY = objectiveFunction(newX)
            if newY > currentY:
                currentY = newY
                currentX = newX
            else:
                prob = r.uniform(0, 1)
                if prob > np.e ** -(currentY - newY)/(heat + (10 ** -10)):
                    currentY = newY
                    currentX = newX
        return(currentX)

sa = SimulatedAnnealing(ranges)
bestX = sa.optimize(griewank)
print(bestX)
print(griewank(bestX))

[np.float64(-119.35948431220896), np.float64(196.18214047742802), np.float64(-118.58995212496117), np.float64(115.13399230041497), np.float64(19.586516612155236), np.float64(27.948044585122126), np.float64(282.1355124511969), np.float64(143.63181716366182), np.float64(227.7469869071787), np.float64(-209.772807642182)]
70.24925192257356


In [5]:
def generateGrid(ranges, nPoints):
    dimensions = len(ranges)
    points = np.zeros((nPoints, dimensions))
    steps = np.ceil(nPoints ** (1 / dimensions)).astype(int)
    for dim, (low, high) in enumerate(ranges):
        linspace = np.linspace(low, high, steps)
        repeats = np.ceil(nPoints / steps).astype(int)
        values = np.tile(linspace, repeats)[:nPoints]
        np.random.shuffle(values)
        points[:, dim] = values
    return(points)

class SecretaryOptimizer:
    def __init__(
        self,
        ranges,
        greedy
    ):
        self.ranges = ranges
        self.greedy = greedy
        
    def optimize(self, objectiveFunction, numIterations = allowedIterations):
        grid = generateGrid(ranges, numIterations)
        np.random.shuffle(grid)
        bestY = float("inf")
        #"Reject" the first N/e secretaries
        for i in range(int(numIterations/np.e)):
            thisX = grid[i]
            thisY = objectiveFunction(thisX)
            if thisY < bestY:
                bestY = thisY
                bestX = thisX
        #"Hire" the first secretary which is better than all these
        j = int(numIterations/np.e)
        exitCondition = False
        while j < numIterations - 1 and not exitCondition:
            #Keep checking secretaries until we find a better one
            j += 1
            thisX = grid[j]
            thisY = objectiveFunction(thisX)
            if thisY < bestY:
                bestY = thisY
                bestX = thisX
                exitCondition = True
        #If we hit exitCondition, we behave like a greedy algorithm - in this case, simulated annealing with low temperature
        bestX = self.greedy.optimize(objectiveFunction, numIterations = numIterations - j)
        return(bestX)

secretary = SecretaryOptimizer(ranges, sa)
bestX = secretary.optimize(griewank)
print(bestX)
print(griewank(bestX))

[np.float64(9.197236319892767), np.float64(-0.7598663729813248), np.float64(-1.9057126472037544), np.float64(3.8913006333441014), np.float64(-4.55319356396732), np.float64(4.743322632193355), np.float64(-8.661032457447009), np.float64(14.867144690697891), np.float64(3.048736318431473), np.float64(-9.80389963236114)]
1.1311393777400427
